# Model 1 on real data: CESAW (REAL, professional developers)

Component 1 · Anchor AI (J26-IT-366). Everything here is made from **REAL** data (CESAW, SEI/CMU), never from the
synthetic students.

**What this notebook answers.** Model 1 learns how far a person's time estimates are off. CESAW is a public record of
about 60,000 real tasks that professional software developers planned and then actually did. If Model 1 predicts
their real overruns better than "trust the plan", that is evidence it works outside our own simulation.

**Limits, in plain words.** These are professional developers, not students. CESAW has no task priority, no task type
and no task category, so every task is one type (`Digital`) with the same priority. The data licence is not stated, so
the data is **not** in the repository: it is downloaded here (Colab) or read from your own copy (`C1_CESAW_PATH`).

**The earned-trust gate.** Model 1 corrects a developer's estimates only after it has *proven* itself on that
developer's own recent tasks (20 most recent, at least 5% closer than their own estimate). Until then the developer
keeps their own estimate; in cold start the model therefore equals B0. The point is the median (q50).

**Cell 1 — setup (run first).** In Google Colab this clones the C1 branch `dev/ravindu` of the team repo into
`/content/AnchorAI` **only if it is not there yet** (running it again never makes a nested copy), goes to that folder,
and installs only what these notebooks need (`components/c1_disruption/notebooks/requirements.txt`: numpy, scipy,
scikit-learn, quantile-forest, joblib, pandas, matplotlib — not the web server). It prints the versions. If a package
that was already loaded had to be replaced (usually numpy), it stops with a clear message: then use
**Runtime → Restart session** and run again from **cell 2** (the next code cell). Outside Colab it uses the repository
the notebook is in and installs nothing. C1 supports Python 3.11–3.13.

In [ ]:
import importlib.metadata as md, os, subprocess, sys

REPO = "https://github.com/DulanjaR/AnchorAI.git"
IN_COLAB = "google.colab" in sys.modules
PACKAGES = {"numpy": "numpy", "scipy": "scipy", "scikit-learn": "sklearn", "quantile-forest": "quantile_forest",
            "joblib": "joblib", "pandas": "pandas", "matplotlib": "matplotlib"}


def repo_root():
    if IN_COLAB:
        return "/content/AnchorAI"
    here = os.path.abspath(os.getcwd())
    while not os.path.isdir(os.path.join(here, "components", "c1_disruption")):
        if os.path.dirname(here) == here:
            raise FileNotFoundError("Open this notebook inside the AnchorAI repository (or in Colab).")
        here = os.path.dirname(here)
    return here


def installed():
    versions = {}
    for package in PACKAGES:
        try:
            versions[package] = md.version(package)
        except md.PackageNotFoundError:
            versions[package] = None
    return versions


ROOT = repo_root()
if IN_COLAB and not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "--depth", "1", "-b", "dev/ravindu", REPO, ROOT], check=True)
elif IN_COLAB:
    print("Already cloned at", ROOT, "- not cloning again.")
os.chdir(ROOT)
before = installed()
loaded = {p: getattr(sys.modules[m], "__version__", None) for p, m in PACKAGES.items() if m in sys.modules}
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    "components/c1_disruption/notebooks/requirements.txt"], check=True)
after = installed()
print("Python", sys.version.split()[0], "| folder", ROOT)
for package, version in after.items():
    print(f"  {package:16} {version}" + ("" if before[package] == version else f"   (was {before[package]})"))
replaced = [p for p, v in loaded.items() if v is not None and v != after[p]]
if replaced:
    print("\n" + "!" * 72 + f"\n{', '.join(replaced)} changed while already loaded.\n"
          "Runtime → Restart session, then run from cell 2 (the next code cell).\n" + "!" * 72)
    raise SystemExit("Restart needed: Runtime → Restart session, then run from cell 2.")

**Cell 2 — start here after a restart.** Goes to the repository folder, makes C1 importable, prints the versions in
use and starts the progress clock (every long step prints a line like `[  12.4s] training fold 3/5…`).

In [ ]:
import os, sys, tempfile

IN_COLAB = "google.colab" in sys.modules
ROOT = "/content/AnchorAI"
if not IN_COLAB:
    ROOT = os.path.abspath(os.getcwd())
    while not os.path.isdir(os.path.join(ROOT, "components", "c1_disruption")):
        ROOT = os.path.dirname(ROOT)
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
# Files this notebook writes go here, never into the repository.
OUT = "/content/c1_outputs" if IN_COLAB else os.path.join(tempfile.gettempdir(), "c1_outputs")
os.makedirs(OUT, exist_ok=True)

import joblib, matplotlib, numpy, pandas, quantile_forest, scipy, sklearn
print("Python", sys.version.split()[0], "| numpy", numpy.__version__, "| scipy", scipy.__version__,
      "| scikit-learn", sklearn.__version__, "| quantile-forest", quantile_forest.__version__,
      "| joblib", joblib.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Outputs go to", OUT)
from components.c1_disruption.ml.progress import Progress
progress = Progress()

**Get the data.** In Colab this downloads `CESAW_task_fact.csv` (about 5.5 MB) to `/content/cesaw` once. Elsewhere,
set the environment variable `C1_CESAW_PATH` to the folder with your copy (outside the repository). Never commit it.

In [ ]:
import subprocess
if IN_COLAB:
    CESAW_PATH = "/content/cesaw"
    if not os.path.isfile(os.path.join(CESAW_PATH, "CESAW_task_fact.csv")):
        os.makedirs(CESAW_PATH, exist_ok=True)
        progress("downloading CESAW_task_fact.csv…")
        subprocess.run(["curl", "-sL", "-o", os.path.join(CESAW_PATH, "CESAW_task_fact.csv"),
                        "https://ndownloader.figshare.com/files/17852276"], check=True)
else:
    CESAW_PATH = os.environ.get("C1_CESAW_PATH", "")
    if not CESAW_PATH:
        raise SystemExit("Set C1_CESAW_PATH to the folder with CESAW_task_fact.csv (outside the repository).")
print("CESAW data:", CESAW_PATH)

**Turn tasks into Model 1 records.** One CESAW task = one record: the developer is the "student", planned minutes are
the estimate, actual minutes the real time (CESAW already leaves out interruptions), the start time gives the hour and
weekday. Tasks with no usable plan, no actual time or no start time are left out and counted.

In [ ]:
from components.c1_disruption.ml.drift.external import cesaw
progress("reading CESAW and mapping tasks to Model 1 records…")
records, dropped = cesaw.load_cesaw(CESAW_PATH)
shape = cesaw.describe(records)
print(dropped)
print(shape)

**How often are real estimates wrong?** log(actual ÷ planned) is 0 when the plan was right, positive when the task ran
over, negative when it was quicker. A wide spread means estimates are very uncertain.

In [ ]:
import numpy as np
from components.c1_disruption.ml.drift.data import log_drift
print("10/25/50/75/90th percentiles of log(actual / planned):", np.percentile(log_drift(records), [10, 25, 50, 75, 90]).round(2))

**Run the test (REAL).** *Cold start* holds groups of developers out completely and predicts them from everyone else;
*warm* predicts a developer's later days from their own earlier days plus everyone else, retraining as it goes.
Each retraining uses about 60,000 tasks, so the full run takes hours on a laptop or Colab. **`QUICK = True` (default)**
runs a short version — 8 developers in the warm part, one retraining per 60 of their days, 30 trees — so the
notebook finishes in reasonable time; its numbers are a preview. The committed results file
`docs/results/c1_model1_cesaw.md` was made with the full settings (`QUICK = False`).

In [ ]:
QUICK = True
if QUICK:
    sample = cesaw.warm_developers(records)[:8]
    report = cesaw.evaluate(records, n_estimators=30, warm_sample=sample, warm_step=60, progress=progress,
                            keep_predictions=True)
else:
    report = cesaw.evaluate(records, progress=progress, keep_predictions=True)
progress("done")
text = cesaw.to_markdown(report, dropped, shape)
print(text)

In [ ]:
from components.c1_disruption.ml import charts
label = "REAL (CESAW, professional developers)" + (", quick run" if QUICK else "")
charts.mae_by_method({m: {"cold start": report["cold"][m]["maeMinutes"], "warm": report["warm"][m]["maeMinutes"]}
                      for m in report["cold"]}, f"Model 1 vs baselines — MAE by method, {label}");

In [ ]:
charts.learning_curve({"Model 1 (warm)": report["learningCurve"]}, f"Model 1 learning curve, {label}");

In [ ]:
import math, random
warm_model = report["predictions"]["warm"]["model"]
points = [(r["estimatedMinutes"], r["actualMinutes"], r["estimatedMinutes"] * math.exp(v))
          for r, v in zip(warm_model.records, warm_model.log_pred)]
points = random.Random(0).sample(points, min(3000, len(points)))  # a readable sample
charts.predicted_vs_actual(points, f"Model 1: predicted vs actual minutes, {label}");

In [ ]:
gate = report["gateByDeveloper"]["warm"]
charts.gate_per_student({d: float(v) for d, v in gate.items()}, f"Earned-trust gate open per developer (warm), {label}");
print(sum(gate.values()), "of", len(gate), "developers have the gate open at their last point")